# Spatial Proteomics-Based Cancer Survival Prediction

**BIOSTAT 626 team research project — computational analysis notebook**

This notebook demonstrates a convolutional neural network (CNN) for feature extraction from 52-channel spatial proteomics images, followed by Random Forest regression to predict overall survival in months. The analysis was originally developed for a course group project; the code here was primarily authored and reorganized by Xueying Zhang.

**Data availability:** The original course-supplied patient data are **not included** in this public code sample. The notebook can only be executed with datasets the user is authorized to access. This is not an independently validated clinical prediction tool.

**Preprocessing note:** The original `/255.0` operation is retained, but each input TIFF is first converted into `uint8` using channel-wise robust percentile scaling. This yields a 0–1 model input while keeping the source TIFF unchanged. Because this rescales each sample independently, it may remove between-sample absolute intensity differences and should not be interpreted as a validated biological normalization procedure.

## 1. Imports, reproducibility, and relative paths

In [ ]:
from pathlib import Path
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from skimage.io import imread
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam

SEED = 10
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# The folder containing this notebook is the project root.
PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data"
TRAIN_CSV = DATA_ROOT / "train" / "train_data.csv"
TRAIN_IMAGE_DIR = DATA_ROOT / "train" / "images"
TEST_IMAGE_DIR = DATA_ROOT / "test" / "images"

required_paths = [TRAIN_CSV, TRAIN_IMAGE_DIR, TEST_IMAGE_DIR]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Place authorized data under data/train/ and data/test/. Missing:\n" + "\n".join(missing)
    )

## 2. Load metadata and check the image format

In [ ]:
train_data = pd.read_csv(TRAIN_CSV)
required_columns = {"id", "OSmonth"}
if not required_columns.issubset(train_data.columns):
    raise ValueError(f"Training CSV must contain {sorted(required_columns)}")
if train_data["id"].duplicated().any():
    raise ValueError("Duplicate patient IDs found")

def image_path(folder, sample_id):
    return folder / f"{int(sample_id)}.tiff"

def load_and_preprocess_image(path):
    image = np.asarray(imread(str(path)))
    if image.shape == (52, 150, 150):
        image = np.transpose(image, (1, 2, 0))
    elif image.shape != (150, 150, 52):
        raise ValueError(f"Unexpected image dimensions {image.shape} for {path}")
    image = image.astype(np.float32)
    if not np.isfinite(image).all():
        raise ValueError(f"Non-finite intensity values in {path}")
    # Convert each protein channel to 8-bit intensity without modifying the TIFF.
    # Robust per-image channel scaling preserves spatial contrasts; it does NOT
    # preserve absolute protein abundance comparisons across different images.
    low = np.percentile(image, 1, axis=(0, 1), keepdims=True)
    high = np.percentile(image, 99, axis=(0, 1), keepdims=True)
    span = np.maximum(high - low, 1e-8)
    image_uint8 = np.rint(np.clip((image - low) / span, 0.0, 1.0) * 255).astype(np.uint8)
    # Original course-project scaling is intentionally preserved.
    image = image_uint8 / 255.0
    return image.astype(np.float32)

first_path = image_path(TRAIN_IMAGE_DIR, train_data.loc[0, "id"])
raw_image = np.asarray(imread(str(first_path)))
print("Raw TIFF dtype:", raw_image.dtype, "shape:", raw_image.shape)
print("Raw intensity range:", float(raw_image.min()), float(raw_image.max()))
print("Preprocessing: per-channel 1st–99th percentile -> uint8 -> /255.0")

## 3. Split patients into training and validation groups

In [ ]:
train_rows, validation_rows = train_test_split(
    train_data, test_size=20, random_state=SEED, shuffle=True
)
if set(train_rows["id"]).intersection(validation_rows["id"]):
    raise AssertionError("Patient IDs overlap between training and validation")

X_train = np.stack([
    load_and_preprocess_image(image_path(TRAIN_IMAGE_DIR, sample_id))
    for sample_id in train_rows["id"]
])
X_validation = np.stack([
    load_and_preprocess_image(image_path(TRAIN_IMAGE_DIR, sample_id))
    for sample_id in validation_rows["id"]
])
y_train = train_rows["OSmonth"].to_numpy(dtype=float)
y_validation = validation_rows["OSmonth"].to_numpy(dtype=float)
print("Training:", X_train.shape, "Validation:", X_validation.shape)

## 4. Train the original CNN regression model

In [ ]:
def create_cnn(input_shape=(150, 150, 52), num_output=1):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation="relu"),
        layers.Flatten(name="feature_vector"),
        layers.Dense(256, activation="relu"),
        layers.Dense(num_output, activation="linear"),
    ])
    return model

cnn_model = create_cnn()
cnn_model.compile(optimizer=Adam(), loss="mean_squared_error", metrics=["mae"])
history = cnn_model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_validation, y_validation),
    verbose=1,
)

## 5. Extract CNN features and fit Random Forest

In [ ]:
feature_extractor = models.Model(
    inputs=cnn_model.inputs,
    outputs=cnn_model.get_layer("feature_vector").output,
)
train_features = feature_extractor.predict(X_train, verbose=0)
validation_features = feature_extractor.predict(X_validation, verbose=0)

rf_regressor = RandomForestRegressor(n_estimators=100, random_state=SEED)
rf_regressor.fit(train_features, y_train)
predictions_validation = rf_regressor.predict(validation_features)

mae = mean_absolute_error(y_validation, predictions_validation)
mse = mean_squared_error(y_validation, predictions_validation)
print(f"Holdout validation MAE: {mae:.3f} months")
print(f"Holdout validation MSE: {mse:.3f} months squared")

## 6. Visualize holdout predictions (no hard-coded results)

In [ ]:
comparison_df = pd.DataFrame({
    "sample_id": validation_rows["id"].to_numpy(),
    "observed_os_months": y_validation,
    "predicted_os_months": predictions_validation,
}).sort_values("sample_id").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(comparison_df["observed_os_months"],
           comparison_df["predicted_os_months"], alpha=0.8)
low = float(min(comparison_df[["observed_os_months", "predicted_os_months"]].min()))
high = float(max(comparison_df[["observed_os_months", "predicted_os_months"]].max()))
ax.plot([low, high], [low, high], linestyle="--", color="gray", linewidth=1)
ax.set(xlabel="Observed survival (months)", ylabel="Predicted survival (months)",
       title=f"Holdout validation (n={len(comparison_df)}): MAE={mae:.1f} months")
fig.tight_layout()
plt.show()

## 7. Apply the trained pipeline to independent test images

In [ ]:
test_image_paths = sorted(
    TEST_IMAGE_DIR.glob("*.tiff"), key=lambda p: int(p.stem)
)
if not test_image_paths:
    raise FileNotFoundError(f"No TIFF files found in {TEST_IMAGE_DIR}")
X_test = np.stack([load_and_preprocess_image(p) for p in test_image_paths])
test_features = feature_extractor.predict(X_test, verbose=0)
predictions_test = rf_regressor.predict(test_features)
test_predictions = pd.DataFrame({
    "sample_id": [int(p.stem) for p in test_image_paths],
    "predicted_os_months": predictions_test,
})
print(f"Generated {len(test_predictions)} held-out test predictions.")
# Original course data and predictions are deliberately not displayed or committed publicly.

## Interpretation and limitations

- This notebook retains the original project's CNN-then-Random-Forest design and `/255` operation, with a newly added `uint8` conversion step before division.
- Per-image channel-wise percentile scaling can remove biologically meaningful between-patient absolute abundance differences. The model is exploratory, and this choice needs biological validation.
- Validation is **one fixed 20-patient holdout split**, not 5-fold cross-validation. Reported error metrics will depend on the actual authorized dataset and software environment.
- A 225-patient cohort is small for CNN training. This is exploratory student research, not a clinical decision tool.
- The publicly shared notebook has **no saved patient-level outputs**. Re-run it only on authorized data; do not push those data or private outputs to GitHub.
- The actual source TIFF dtype may be floating point; the **model inputs** are constructed from an intermediate `uint8` representation. Source files are never modified.